# SageAgent v5 Production UI

SageAgent v5 is the production-ready SageMaker notebook coding agent. It keeps the v4-style notebook UI, but runs the rebuilt v5 engine underneath: Bedrock Claude models, durable memory/status, subagents, checkpoints, compaction, result replay, telemetry, and verify/done gates.

**Run cells 1-3 in order:**
- **Cell 1** installs packages, usually once per kernel.
- **Cell 2** sets model, AWS region, mock mode, thinking mode, and budget settings.
- **Cell 3** launches the chat UI.

**Core runtime files in the production zip:** `chat.ipynb`, `entry.py`, `sagemaker_agent.py`, `agent.py`, `commands.py`, `memory.md`, `AGENT_STATUS.md`, and the `core/`, `runtime/`, `tools/`, `prompt/`, `skills/`, `subagent/`, `security/`, and `ui/` packages.

**Companion guide:** open `chat.md` for the user guide, command list, skills list, safety notes, and troubleshooting.

In [ ]:
# Cell 1: install dependencies (run once per kernel)
!pip install -q boto3 ipywidgets jupyterlab_widgets widgetsnbextension Pillow python-docx openpyxl matplotlib requests scikit-learn


In [ ]:
# Locate the v5 runtime whether Jupyter starts from the repo root, compact_v5, or the shipped zip root.
import sys
from pathlib import Path

def _ensure_sageagent_path():
    start = Path.cwd().resolve()
    candidates = []
    for base in (start, *start.parents):
        # Prefer source layout first, then shipped flat-zip layout.
        candidates.extend([
            base / "MAIN" / "agent",
            base / "compact_v5" / "MAIN" / "agent",
            base / "sagemaker-coding-agent" / "compact_v5" / "MAIN" / "agent",
            base,
            base / "compact_v5",
            base / "sagemaker-coding-agent" / "compact_v5",
        ])
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "entry.py").is_file():
            path = str(candidate)
            if path not in sys.path:
                sys.path.insert(0, path)
            return candidate
    raise ModuleNotFoundError(
        "Could not locate entry.py. Open chat.ipynb from the shipped zip root "
        "or compact_v5/MAIN/agent, or set PYTHONPATH to the folder containing entry.py."
    )

_AGENT_ROOT = _ensure_sageagent_path()
print(f"Using SageAgent runtime: {_AGENT_ROOT}")

# Cell 2: configure with v4-style controls
from entry import CONFIG, BEDROCK_MODELS

AVAILABLE_MODELS = dict(BEDROCK_MODELS)
TEMPERATURE_OPTIONS = {
    "0.0 - Deterministic": 0.0,
    "0.3 - Low creativity": 0.3,
    "0.5 - Balanced": 0.5,
    "0.7 - High creativity": 0.7,
    "1.0 - Maximum creativity": 1.0,
}
THINKING_BUDGET_OPTIONS = {
    "1024 - Minimal": 1024,
    "2048 - Light": 2048,
    "4096 - Standard": 4096,
    "8192 - Extended": 8192,
    "16000 - Maximum": 16000,
}
REGION = "ap-southeast-2"  # Sydney, matching v4

def _label_for_model(model_id):
    for label, mid in BEDROCK_MODELS:
        if mid == model_id:
            return label
    return BEDROCK_MODELS[0][0]

def _value(widget_or_value):
    return getattr(widget_or_value, "value", widget_or_value)

# v4-compatible defaults: first model is Claude 4.5 Sonnet (AU).
default_model_name = _label_for_model(BEDROCK_MODELS[0][1])

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML
    _WIDGETS_OK = True
except Exception:
    widgets = None
    display = None
    HTML = None
    _WIDGETS_OK = False

if _WIDGETS_OK:
    display(HTML("<h3>Agent Configuration</h3>"))
    model_dropdown = widgets.Dropdown(
        options=list(AVAILABLE_MODELS.keys()),
        value=default_model_name,
        description="Model:",
        style={"description_width": "120px"},
        layout=widgets.Layout(width="520px"),
    )
    temperature_dropdown = widgets.Dropdown(
        options=list(TEMPERATURE_OPTIONS.keys()),
        value="0.0 - Deterministic",
        description="Temperature:",
        style={"description_width": "120px"},
        layout=widgets.Layout(width="360px"),
    )
    thinking_budget_dropdown = widgets.Dropdown(
        options=list(THINKING_BUDGET_OPTIONS.keys()),
        value="4096 - Standard",
        description="Thinking Budget:",
        style={"description_width": "120px"},
        layout=widgets.Layout(width="360px"),
    )
    max_turns_slider = widgets.IntSlider(
        value=60, min=5, max=100, step=5,
        description="Max Turns:",
        style={"description_width": "120px"},
        layout=widgets.Layout(width="400px"),
    )
    iteration_budget_slider = widgets.IntSlider(
        value=600, min=90, max=2000, step=50,
        description="Iter Budget:",
        style={"description_width": "120px"},
        layout=widgets.Layout(width="400px"),
    )
    workspace_input = widgets.Text(
        value=".",
        description="Workspace:",
        placeholder="Directory for file operations",
        style={"description_width": "120px"},
        layout=widgets.Layout(width="400px"),
    )
    mock_toggle = widgets.Checkbox(
        value=False,
        description="Mock Mode (test without API)",
        indent=False,
    )
    thinking_checkbox = widgets.Checkbox(
        value=False,
        description="Enable Extended Thinking (slower, uses more tokens)",
        indent=False,
    )
    bedrock_only_toggle = widgets.Checkbox(
        value=True,
        description="Bedrock-only (block S3, Lambda, Textract, etc.)",
        indent=False,
    )
    config_box = widgets.VBox([
        model_dropdown,
        widgets.HTML(f"<p style='margin:5px 0;color:#888;'>Region: Sydney ({REGION})</p>"),
        temperature_dropdown,
        thinking_checkbox,
        thinking_budget_dropdown,
        workspace_input,
        max_turns_slider,
        iteration_budget_slider,
        mock_toggle,
        bedrock_only_toggle,
    ], layout=widgets.Layout(
        padding="10px",
        border="1px solid #444",
        margin="10px 0",
        background="#2d2d2d",
    ))
    display(config_box)
    display(HTML("<p style='color:#888;font-size:12px;'>Configure settings above, then run the next cell to start.</p>"))
else:
    class _Value:
        def __init__(self, value):
            self.value = value
    model_dropdown = _Value(default_model_name)
    temperature_dropdown = _Value("0.0 - Deterministic")
    thinking_budget_dropdown = _Value("4096 - Standard")
    max_turns_slider = _Value(60)
    iteration_budget_slider = _Value(600)
    workspace_input = _Value(".")
    mock_toggle = _Value(True)
    thinking_checkbox = _Value(False)
    bedrock_only_toggle = _Value(True)

# Apply defaults immediately so Cell 2 is useful even before Cell 3.
CONFIG.model_id = AVAILABLE_MODELS[_value(model_dropdown)]
CONFIG.region = REGION
CONFIG.workspace = _value(workspace_input)
CONFIG.max_turns = int(_value(max_turns_slider))
CONFIG.max_iteration_budget = int(_value(iteration_budget_slider))
CONFIG.mock_mode = bool(_value(mock_toggle))
CONFIG.temperature = TEMPERATURE_OPTIONS[_value(temperature_dropdown)]
CONFIG.thinking_enabled = bool(_value(thinking_checkbox))
CONFIG.thinking_budget = THINKING_BUDGET_OPTIONS[_value(thinking_budget_dropdown)]
CONFIG.require_tool_approval = True
CONFIG.aws_bedrock_only = bool(_value(bedrock_only_toggle))
CONFIG.session_cost_limit = 5.0
CONFIG.enable_skill_auto_trigger = False

print(
    f"Configured: model={CONFIG.model_id}, region={CONFIG.region}, "
    f"mock_mode={CONFIG.mock_mode}, thinking={CONFIG.thinking_enabled}, "
    f"iter_budget={CONFIG.max_iteration_budget}"
)


In [ ]:
# Locate the v5 runtime whether Jupyter starts from the repo root, compact_v5, or the shipped zip root.
import sys
from pathlib import Path

def _ensure_sageagent_path():
    start = Path.cwd().resolve()
    candidates = []
    for base in (start, *start.parents):
        # Prefer source layout first, then shipped flat-zip layout.
        candidates.extend([
            base / "MAIN" / "agent",
            base / "compact_v5" / "MAIN" / "agent",
            base / "sagemaker-coding-agent" / "compact_v5" / "MAIN" / "agent",
            base,
            base / "compact_v5",
            base / "sagemaker-coding-agent" / "compact_v5",
        ])
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "entry.py").is_file():
            path = str(candidate)
            if path not in sys.path:
                sys.path.insert(0, path)
            return candidate
    raise ModuleNotFoundError(
        "Could not locate entry.py. Open chat.ipynb from the shipped zip root "
        "or compact_v5/MAIN/agent, or set PYTHONPATH to the folder containing entry.py."
    )

_AGENT_ROOT = _ensure_sageagent_path()
print(f"Using SageAgent runtime: {_AGENT_ROOT}")

# Cell 3: launch chat UI
from entry import CONFIG
from entry import create_chat_ui
from IPython.display import display

# Re-apply widget selections from Cell 2 just before launch.
try:
    CONFIG.model_id = AVAILABLE_MODELS[_value(model_dropdown)]
    CONFIG.region = REGION
    CONFIG.workspace = _value(workspace_input)
    CONFIG.max_turns = int(_value(max_turns_slider))
    CONFIG.max_iteration_budget = int(_value(iteration_budget_slider))
    CONFIG.mock_mode = bool(_value(mock_toggle))
    CONFIG.temperature = TEMPERATURE_OPTIONS[_value(temperature_dropdown)]
    CONFIG.thinking_enabled = bool(_value(thinking_checkbox))
    CONFIG.thinking_budget = THINKING_BUDGET_OPTIONS[_value(thinking_budget_dropdown)]
    CONFIG.aws_bedrock_only = bool(_value(bedrock_only_toggle))
except NameError:
    print("Cell 2 controls were not found; using existing CONFIG values.")

ui = create_chat_ui()
# v5 follows compact_v4's stable display contract: create_chat_ui()
# displays one complete top-level widget internally and returns `ui`
# as the programmatic handle. Do not display child widgets separately.


---

## Quick Reference

**Buttons:** Send, Stop, Clear, Compact, Clean. If widgets are unavailable, use the console fallback: `ui.send("your message")`.

**Long-running work flow:** keep `AGENT_STATUS.md` and `memory.md` fresh, use `/save`, `/resume`, and `/checkpoint` for handoff/rollback, then use `/verify` and `/done` before trusting a completed software task.

**Core commands:** `/status`, `/save`, `/resume`, `/checkpoint`, `/cost`, `/context`, `/verify`, `/done`, `/dream`, `/skills`, `/skill use`, `/skill clear`, `/skill apply`, `/skill reject`, `/skillify`, `/promote-to-skill`, `/phase`, `/diffs`, `/regression`, `/revert`, `/auth`, `/quit`.

**Production skills:** batch, clara, debug, design, html, init, init-verifiers, reflexion, remember, report, review, security-review, simplify, skillify, verify. Use `/skills` to list them and `/skill use <name>` to activate one.

**Cost and context:** `/cost` shows token/cache/model spend. `/context` shows context pressure. v5 tracks parent/subagent usage and blocks unsafe done claims through the verification gates.

**Memory:** `memory.md` is auto-loaded as durable memory. `/dream` consolidates memory when you want a cleanup pass.

**Ship evidence:** v5.0.1 passed the final R-tier gate and final Claude production-readiness review. Test evidence is kept outside the production zip under `_status/`.
